# Libraries and definition of functions and layers for EF-CNN architectures

In [1]:
# libraries
import tensorflow as tf
import numpy as np

###################################### DEFINITION OF AUXLIAR FUNCTIONS ######################################

# image processing for gray scale images (FFT, shift to build centered representation and generation of reduced representation)
def FFT_rect(images, shift=False, pad=False, out_dim=None, expand_dims=True):
    outs = []
    # iterate over each image
    for img in images:
        # perform FFT
        aux = np.fft.fft2(img)

        # shift image to build centered FFT
        if shift: # representacion centrada
            aux = np.fft.fftshift(aux)

        # transform output to rectangular complex representation
        if pad:
            outs.append([np.pad(np.real(aux),2,'constant'), np.pad(np.imag(aux),2,'constant')])
        else:
            outs.append([np.real(aux), np.imag(aux)])
    outs = np.array(outs, dtype=np.float32)

    # generation of reduced representation
    if out_dim != None:
        r_mid = int(images.shape[1]/2)
        c_mid = int(images.shape[2]/2)
        dl = int(out_dim/2)
        outs = outs[:,:,r_mid-dl:r_mid+dl,c_mid-dl:c_mid+dl]

    # expansion of dimensions
    if expand_dims:
        outs = np.expand_dims(outs, axis=-1)

    return outs


# image processing for RGB images (FFT, shift to build centered representation and generation of reduced representation)
def FFT_rect_3d(images, shift=False, out_dim=None):
    outs = []
    # iterate over each image
    for img in images:
        fft = []
        # iterate over each dimension
        for i in range(3):
            ffti = np.fft.fft2(img[:,:,i])
            if shift:
                ffti = np.fft.fftshift(ffti)
            fft.append(ffti)
        fft = np.array(fft)
        fft = np.swapaxes(fft, 0,2)
        fft = np.swapaxes(fft, 0,1)
        outs.append([np.real(fft), np.imag(fft)])

    Xs = np.array(outs, dtype=np.float32)

    # generation of reduced representation
    if out_dim != None:
        mid = int(Xs.shape[2]/2)
        dx = int(out_dim/2)
        Xs = Xs[:,:,mid-dx:mid+dx,mid-dx:mid+dx,:]

    return Xs


# definition of auxiliar class to train the model and save the configuration of the beast epoch
class MaxEpoch(tf.keras.callbacks.Callback):
    def __init__(self, epochs):
        super().__init__()
        self.epochs = epochs # number of training epochs

        # auxliar variables
        self.max_epoch = 0
        self.max_val_acc = 0.0
        self.max_weights = None

    def on_epoch_end(self, epoch, logs=None):
        # get validation accuracy
        val_acc = logs.get('val_acc')

        # update max epoch
        if val_acc > self.max_val_acc:
            self.max_epoch = epoch
            self.max_val_acc = val_acc
            self.max_weights = self.model.get_weights()

        return super().on_epoch_end(epoch, logs)

    def on_train_end(self, logs=None):
        return super().on_train_end(logs)




###################################### DEFINITION OF ACTIVATION FUNCTIONS ######################################

class CReLU(tf.keras.layers.Layer):
    def __init__(self):
        super().__init__()

    def build(self, input_shape):
        return super().build(input_shape)

    def call(self, inputs):
        res = inputs[:,0]
        ims = inputs[:,1]

        return tf.stack([tf.nn.relu(res), tf.nn.relu(ims)], axis=1)


class modReLU(tf.keras.layers.Layer):
    def __init__(self, filters):
        super().__init__()
        self.filters = filters

    def build(self, input_shape):
        self.b = self.add_weight(shape=(self.filters,), name='bias',
                                  initializer=tf.keras.initializers.zeros(), trainable=True)

    def call(self, inputs):
        res = inputs[:,0]
        ims = inputs[:,1]

        coms = tf.complex(res,ims)
        mag = tf.math.abs(coms)
        pha = tf.math.angle(coms)

        rmag = tf.nn.relu(mag + self.b)

        return tf.stack([rmag*tf.math.cos(pha), rmag*tf.sin(pha)], axis=1)




###################################### DEFINITION OF LINEAR TRANSFORM LAYERS ######################################

# definition of Linear Transform layer for gray scale images
class LinearTqs(tf.keras.layers.Layer):
    def __init__(self, out_dim):
        super().__init__()
        self.out_dim = out_dim

    def get_config(self):
        return {"out_dim": self.out_dim}

    def build(self, input_shape):
        # calculation of the center of the input
        self.rm = int(input_shape[2]/2)
        self.cm = int(input_shape[3]/2)

        # split on quadrants
        self.resh_flatt = tf.keras.layers.Reshape(target_shape=(2, 4, 1, self.rm*self.cm))

        # definition of parameters
        self.mt = self.add_weight(shape=(4, int(input_shape[2]/2)*int(input_shape[3]/2), int(self.out_dim/2)**2), initializer='random_normal', trainable=True)

        # definition of reshape layer
        self.resh_cs = tf.keras.layers.Reshape(target_shape=(2, 4, int(self.out_dim/2), int(self.out_dim/2), 1))

    def call(self, x): # (batch, cord, rows, cols)
        # split of information in quadrants
        q1 = x[:,:,:self.rm,:self.cm]
        q2 = x[:,:,:self.rm,self.cm:]
        q3 = x[:,:,self.rm:,:self.cm]
        q4 = x[:,:,self.rm:,self.cm:]
        qs = tf.stack([q1,q2,q3,q4], axis=2)

        # flatten representation
        flats = self.resh_flatt(qs)

        # linear transform operation
        mts = tf.matmul(flats, self.mt)
        chs = self.resh_cs(mts)

        # concatenation of quadrants
        up = tf.concat([chs[:,:,0], chs[:,:,1]], axis=3)
        down = tf.concat([chs[:,:,2], chs[:,:,3]], axis=3)
        ys = tf.concat([up, down], axis=2)

        return ys


# definition of Linear Transform layer for RGB images
class LinearTqsRGB(tf.keras.layers.Layer):
    def __init__(self, out_dim):
        super().__init__()
        self.out_dim = out_dim

    def get_config(self):
        return {"out_dim": self.out_dim}

    def build(self, input_shape):
        self.rm = int(input_shape[2]/2)
        self.cm = int(input_shape[3]/2)

        # definition of reshape layers
        self.resh_flatt1 = tf.keras.layers.Reshape(target_shape=(2, 4, 1, self.rm*self.cm))
        self.resh_flatt2 = tf.keras.layers.Reshape(target_shape=(2, 4, 1, self.rm*self.cm))
        self.resh_flatt3 = tf.keras.layers.Reshape(target_shape=(2, 4, 1, self.rm*self.cm))

        # definition of parameters
        self.mt = self.add_weight(shape=(4, int(input_shape[2]/2)*int(input_shape[3]/2), int(self.out_dim/2)**2), initializer='random_normal', trainable=True)

        # reshape layers
        self.resh_cs1 = tf.keras.layers.Reshape(target_shape=(2, 4, int(self.out_dim/2), int(self.out_dim/2), 1))
        self.resh_cs2 = tf.keras.layers.Reshape(target_shape=(2, 4, int(self.out_dim/2), int(self.out_dim/2), 1))
        self.resh_cs3 = tf.keras.layers.Reshape(target_shape=(2, 4, int(self.out_dim/2), int(self.out_dim/2), 1))

    def call(self, x):
        # split in quadrants by color channel
        q11 = x[:,:,:self.rm,:self.cm,0]
        q12 = x[:,:,:self.rm,self.cm:,0]
        q13 = x[:,:,self.rm:,:self.cm,0]
        q14 = x[:,:,self.rm:,self.cm:,0]
        q1s = tf.stack([q11,q12,q13,q14], axis=2)

        q21 = x[:,:,:self.rm,:self.cm,1]
        q22 = x[:,:,:self.rm,self.cm:,1]
        q23 = x[:,:,self.rm:,:self.cm,1]
        q24 = x[:,:,self.rm:,self.cm:,1]
        q2s = tf.stack([q21,q22,q23,q24], axis=2)

        q31 = x[:,:,:self.rm,:self.cm,2]
        q32 = x[:,:,:self.rm,self.cm:,2]
        q33 = x[:,:,self.rm:,:self.cm,2]
        q34 = x[:,:,self.rm:,self.cm:,2]
        q3s = tf.stack([q31,q32,q33,q34], axis=2)

        # flatten transformation
        flats1 = self.resh_flatt1(q1s)
        flats2 = self.resh_flatt2(q2s)
        flats3 = self.resh_flatt3(q3s)

        # linear transform operation
        mts1 = tf.matmul(flats1, self.mt)
        chs1 = self.resh_cs1(mts1)
        mts2 = tf.matmul(flats2, self.mt)
        chs2 = self.resh_cs2(mts2)
        mts3 = tf.matmul(flats3, self.mt)
        chs3 = self.resh_cs3(mts3)

        # concatenation
        up1 = tf.concat([chs1[:,:,0], chs1[:,:,1]], axis=3)
        down1 = tf.concat([chs1[:,:,2], chs1[:,:,3]], axis=3)
        ys1 = tf.concat([up1, down1], axis=2)
        up2 = tf.concat([chs2[:,:,0], chs2[:,:,1]], axis=3)
        down2 = tf.concat([chs2[:,:,2], chs2[:,:,3]], axis=3)
        ys2 = tf.concat([up2, down2], axis=2)
        up3 = tf.concat([chs3[:,:,0], chs3[:,:,1]], axis=3)
        down3 = tf.concat([chs3[:,:,2], chs3[:,:,3]], axis=3)
        ys3 = tf.concat([up3, down3], axis=2)

        ys = tf.concat([ys1, ys2, ys3], axis=-1)

        return ys


###################################### DEFINITION OF EF-CNN LAYERS ######################################

# definition of layer to perform Hadamard product using random kernels
class RandomKernels(tf.keras.layers.Layer):
    def __init__(self, filters, act='relu'):
        super().__init__()
        self.filters = filters

        # activation function
        self.act = CReLU()

        if act == 'crelu':
            self.act = CReLU()
        elif act == 'modrelu':
            self.act = modReLU(filters=self.filters)
        elif act == 'identity':
            self.act = tf.identity

    def get_config(self):
        return {"filters": self.filters, "act": self.act}

    def build(self, input_shape):
        # definition of random kernels
        self.W_r = self.add_weight(
            shape=(input_shape[-3], input_shape[-2], input_shape[-1], self.filters),
            initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.05), trainable=False)
        self.W_i = self.add_weight(
            shape=(input_shape[-3], input_shape[-2], input_shape[-1], self.filters),
            initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.05), trainable=False)

        # definition of trainable parameters
        self.es_r = self.add_weight(shape=(input_shape[-1], self.filters), initializer="ones", trainable=True)
        self.es_i = self.add_weight(shape=(input_shape[-1], self.filters), initializer="ones", trainable=True)

    def call(self, x):
        # add extra dimension
        x = tf.expand_dims(x, axis=-1)

        # Hadamard product (real and imaginary part)
        r = x[:,0]*self.W_r - x[:,1]*self.W_i
        r = r*self.es_r
        r = tf.reduce_sum(r, axis=3)

        i = x[:,0]*self.W_i + x[:,1]*self.W_r
        i = i*self.es_i
        i = tf.reduce_sum(i, axis=3)

        # concatenation of coordinates
        y = tf.stack([r, i], axis=1)

        # evaluation with activation functions
        return self.act(y)


# definition of layer to calculate the magnitude of the frequency representation
class Magnitude(tf.keras.layers.Layer):
    def __init__(self):
        super().__init__()

    def get_config(self):
        return {}

    def build(self, input_shape):
        pass

    def call (self, inputs):
        re = inputs[:,0,:,:,:]
        im = inputs[:,1,:,:,:]

        out = re*re + im*im

        return out

## R-EF-CNN

In [2]:
# load fashion-mnist dataset
fmnist = tf.keras.datasets.fashion_mnist
(train_images, train_labels), (test_images, test_labels) = fmnist.load_data()

# concatenation of train and test sets
data = np.concatenate([train_images, test_images], axis=0)
labels = np.concatenate([train_labels, test_labels], axis=0)

# generation of rectangular centered FFT (reduced representation of size 4x4)
data = FFT_rect(data, shift=True, out_dim=4, expand_dims=True)

# generacion of train (70%), validation (10%) and test (20%) sets
inds = np.arange(0, len(data), 1, dtype=np.int32)
np.random.seed(1) # random seed
np.random.shuffle(inds)

inds_train = inds[0:int(0.7*len(data))]
inds_val = inds[int(0.7*len(data)): int(0.8*len(data))]
inds_test = inds[int(0.8*len(data)):]

X_train = data[inds_train]
y_train = labels[inds_train]
X_val = data[inds_val]
y_val = labels[inds_val]
X_test = data[inds_test]
y_test = labels[inds_test]

data = None
labels = None
train_images = None
test_images = None
inds_train = None
inds_test = None
inds_val = None

X_train.shape, X_val.shape, X_test.shape

((49000, 2, 4, 4, 1), (7000, 2, 4, 4, 1), (14000, 2, 4, 4, 1))

In [3]:
# definition of hyperparameters
epochs = 50
lr = 0.0005
momentum = 0.9
batch_size = 128

# model definition
inputs = tf.keras.Input((2,4,4,1))

conv1 = RandomKernels(filters=6, act='crelu')(inputs)

conv2 = RandomKernels(filters=16, act='crelu')(conv1)

conv3 = RandomKernels(filters=120, act='crelu')(conv2)

mag = Magnitude()(conv3)

flat = tf.keras.layers.Flatten()(mag)
bn = tf.keras.layers.BatchNormalization()(flat)
dense = tf.keras.layers.Dense(84, activation='relu')(bn)
outputs = tf.keras.layers.Dense(10, activation='softmax')(dense)

model = tf.keras.Model(inputs, outputs)

# print model summary
model.summary()

# compilation
model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=lr, momentum=momentum),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["acc"]
)

# callback definition
callback = MaxEpoch(epochs=epochs)

# fit with traning data
metrici = model.fit(X_train, y_train, batch_size=batch_size, epochs=epochs,  validation_data=(X_val, y_val),
                    callbacks=callback, shuffle=True)

# model evaluation over test set
model_max = tf.keras.models.clone_model(model)
model_max.set_weights(callback.max_weights)
m = tf.keras.metrics.CategoricalAccuracy()
m.reset_state()
m.update_state(tf.one_hot(y_test, depth=10), model_max.predict(X_test))
test_max_acc = m.result().numpy()
print(f'\n---- MAX EPOCH (VAL): {callback.max_epoch+1} TEST_ACC: {test_max_acc} ----')

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 2, 4, 4, 1)]      0         
                                                                 
 random_kernels (RandomKerne  (None, 2, 4, 4, 6)       204       
 ls)                                                             
                                                                 
 random_kernels_1 (RandomKer  (None, 2, 4, 4, 16)      3264      
 nels)                                                           
                                                                 
 random_kernels_2 (RandomKer  (None, 2, 4, 4, 120)     65280     
 nels)                                                           
                                                                 
 magnitude (Magnitude)       (None, 4, 4, 120)         0         
                                                             

## LT-EF-CNN

In [4]:
# load fashion-mnist dataset
fmnist = tf.keras.datasets.fashion_mnist
(train_images, train_labels), (test_images, test_labels) = fmnist.load_data()

# concatenation of train and test sets
data = np.concatenate([train_images, test_images], axis=0)
labels = np.concatenate([train_labels, test_labels], axis=0)

# generation of rectangular centered FFT
data = FFT_rect(data, shift=True, expand_dims=False)

# generacion of train (70%), validation (10%) and test (20%) sets
inds = np.arange(0, len(data), 1, dtype=np.int32)
np.random.seed(1) # random seed
np.random.shuffle(inds)

inds_train = inds[0:int(0.7*len(data))]
inds_val = inds[int(0.7*len(data)): int(0.8*len(data))]
inds_test = inds[int(0.8*len(data)):]

X_train = data[inds_train]
y_train = labels[inds_train]
X_val = data[inds_val]
y_val = labels[inds_val]
X_test = data[inds_test]
y_test = labels[inds_test]

data = None
labels = None
train_images = None
test_images = None
inds_train = None
inds_test = None
inds_val = None

X_train.shape, X_val.shape, X_test.shape

((49000, 2, 28, 28), (7000, 2, 28, 28), (14000, 2, 28, 28))

In [ ]:
# definition of hyperparameters
epochs = 50
lr = 0.0005
momentum = 0.9
batch_size = 128

# model definition
inputs = tf.keras.Input((2,28,28))

lt = LinearTqs(out_dim=4)(inputs)

conv1 = RandomKernels(filters=6, act='crelu')(lt)

conv2 = RandomKernels(filters=16, act='crelu')(conv1)

conv3 = RandomKernels(filters=120, act='crelu')(conv2)

mag = Magnitude()(conv3)

flat = tf.keras.layers.Flatten()(mag)
bn = tf.keras.layers.BatchNormalization()(flat)
dense = tf.keras.layers.Dense(84, activation='relu')(bn)
outputs = tf.keras.layers.Dense(10, activation='softmax')(dense)

model = tf.keras.Model(inputs, outputs)

# print model summary
model.summary()

# compilation
model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=lr, momentum=momentum),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["acc"]
)

# callback definition
callback = MaxEpoch(epochs=epochs)

# fit with traning data
metrici = model.fit(X_train, y_train, batch_size=batch_size, epochs=epochs,  validation_data=(X_val, y_val),
                    callbacks=callback, shuffle=True)

# model evaluation over test set
model_max = tf.keras.models.clone_model(model)
model_max.set_weights(callback.max_weights)
m = tf.keras.metrics.CategoricalAccuracy()
m.reset_state()
m.update_state(tf.one_hot(y_test, depth=10), model_max.predict(X_test))
test_max_acc = m.result().numpy()
print(f'\n---- MAX EPOCH (VAL): {callback.max_epoch+1} TEST_ACC: {test_max_acc} ----') 

Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 2, 28, 28)]       0         
                                                                 
 linear_tqs (LinearTqs)      (None, 2, 4, 4, 1)        3136      
                                                                 
 random_kernels_6 (RandomKer  (None, 2, 4, 4, 6)       204       
 nels)                                                           
                                                                 
 random_kernels_7 (RandomKer  (None, 2, 4, 4, 16)      3264      
 nels)                                                           
                                                                 
 random_kernels_8 (RandomKer  (None, 2, 4, 4, 120)     65280     
 nels)                                                           
                                                           